In [1]:
import h5py
import numpy as np

file_path = '/jizhicfs/easyluwu/Eddy/2D_model/hrkz-torchqg-be45912/output/geo_dump.h5'

with h5py.File(file_path, 'r') as f:
    # 读取时间序列
    time = np.array(f['time'])
    print(f"时间序列长度: {len(time)}")
    print(f"时间示例: {time[:5]}")  # 显示前5个时间点

    # 读取 \mathcal{F}_p 和 \mathcal{F}_q 数据集
    F_p = np.array(f[r'\mathcal{F}_p'])  
    F_q = np.array(f[r'\mathcal{F}_q'])

    # 查看数据维度
    print(f"\n\\mathcal{{F}}_p 维度: {F_p.shape}")
    print(f"\\mathcal{{F}}_q 维度: {F_q.shape}")

    # 如果是时空场数据（例如：时间 × 纬度 × 经度）
    if len(F_p.shape) == 3:
        print("\n时空场数据示例：")
        print(f"时间步数量: {F_p.shape[0]}")
        print(f"空间维度: {F_p.shape[1]}x{F_p.shape[2]}")
        print(f"第一个时间步的统计信息:")
        print(f"  - 最小值: {F_p[0].min():.3e}")
        print(f"  - 最大值: {F_p[0].max():.3e}")
        print(f"  - 均值: {F_p[0].mean():.3e}")

    # 验证时间步一致性
    if len(time) == F_p.shape[0]:
        print("\n时间步与场数据维度一致")
    else:
        print("\n警告：时间序列长度与场数据时间步不一致！")

时间序列长度: 10000
时间示例: [0.0004 0.002  0.0036 0.0052 0.0068]

\mathcal{F}_p 维度: (10000, 512, 512)
\mathcal{F}_q 维度: (10000, 512, 512)

时空场数据示例：
时间步数量: 10000
空间维度: 512x512
第一个时间步的统计信息:
  - 最小值: -9.616e-02
  - 最大值: 1.209e-01
  - 均值: -5.082e-21

时间步与场数据维度一致


In [7]:
import numpy as np
import netCDF4 as nc
import torch
import torch.utils.data as data
import h5py



class train_Dataset(data.Dataset):
    def __init__(self):
        super(train_Dataset, self).__init__()
        self.indices = range(3000, 8000, 1)
        
    def __getitem__(self, index):
        idx = self.indices[index]
        file_path = '/jizhicfs/easyluwu/Eddy/2D_model/hrkz-torchqg-be45912/output/geo_dump.h5'
        with h5py.File(file_path, 'r') as file:
            input_p = file['\\mathcal{F}_p'][idx]
            input_q = file['\\mathcal{F}_q'][idx]
            target_p = file['\\mathcal{F}_p'][idx+100]
            target_q = file['\\mathcal{F}_q'][idx+100]

        input = np.stack([input_p, input_q], 0)
        target = np.stack([target_p, target_q], 0)
        
        input = torch.tensor(input)
        target = torch.tensor(target)
        input = torch.nan_to_num(input, nan=0.0)
        target = torch.nan_to_num(target, nan=0.0)
        
        return input, target

    def __len__(self):
        return len(self.indices)
    
class test_Dataset(data.Dataset):
    def __init__(self):
        super(test_Dataset, self).__init__()
        self.indices = range(8000, 9800, 1)
        
    def __getitem__(self, index):
        idx = self.indices[index]
        file_path = '/jizhicfs/easyluwu/Eddy/2D_model/hrkz-torchqg-be45912/output/geo_dump.h5'
        with h5py.File(file_path, 'r') as file:
            input_p = file['\\mathcal{F}_p'][idx]
            input_q = file['\\mathcal{F}_q'][idx]
            target_p = file['\\mathcal{F}_p'][idx+100]
            target_q = file['\\mathcal{F}_q'][idx+100]

        input = np.stack([input_p, input_q], 0)
        target = np.stack([target_p, target_q], 0)
        
        input = torch.tensor(input)
        target = torch.tensor(target)
        input = torch.nan_to_num(input, nan=0.0)
        target = torch.nan_to_num(target, nan=0.0)
        
        return input, target

    def __len__(self):
        return len(self.indices)

